In [2]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import RFECV, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

from lightgbm import LGBMClassifier
from mrmr import mrmr_classif
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
X_train = pd.read_excel("Data/X_train.xlsx").iloc[:, 1:]
y_train = pd.read_excel("Data/y_train.xlsx").iloc[:, 1].values
X_test  = pd.read_excel("Data/X_test.xlsx").iloc[:, 1:]
y_test  = pd.read_excel("Data/y_test.xlsx").iloc[:, 1].values

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

In [4]:
class MRMRSelector(BaseEstimator, TransformerMixin):
    def __init__(self, K=10):
        self.K = K
        self.selected_features_ = None

    def fit(self, X, y):
        # Convert through numpy first to guarantee a clean 0-based RangeIndex on
        # both X_df and y_s. Without this, CV-sliced DataFrames keep their original
        # non-contiguous index while pd.Series(y) gets 0..n, causing mrmr's internal
        # boolean indexer to raise IndexingError on index misalignment.
        Xarr = np.array(X)
        cols = getattr(X, "columns", range(Xarr.shape[1]))
        X_df = pd.DataFrame(Xarr, columns=cols)
        y_s  = pd.Series(np.array(y).ravel())
        self.selected_features_ = mrmr_classif(X_df, y_s, K=self.K)
        return self

    def transform(self, X):
        Xarr = np.array(X)
        cols = getattr(X, "columns", range(Xarr.shape[1]))
        X_df = pd.DataFrame(Xarr, columns=cols)
        return X_df[self.selected_features_].values

In [5]:
models = {
    "XGB": XGBClassifier(eval_metric="logloss", nthread=1),
    "LR": LogisticRegression(max_iter=2000),
    "RF": RandomForestClassifier(n_jobs=1),
    "MLP": MLPClassifier(early_stopping=True),
    "SVM": SVC(probability=True, cache_size=2000),
    "AB": AdaBoostClassifier(),
    "ET": ExtraTreesClassifier(n_jobs=1),
    # min_child_samples>=5 prevents -inf gain loops on small leaves.
    # force_col_wise avoids a Windows threading deadlock in LightGBM.
    "LGBM": LGBMClassifier(verbose=-1, n_jobs=1, min_child_samples=5,
                   min_split_gain=1e-4, force_col_wise=True)
}

param_grids = {
    "XGB": {'model__max_depth':[2,3], 'model__eta':[0.01,0.03,0.3], 'model__n_estimators':[30,50,100], 'model__reg_lambda':[1,3,8]},
    "LR": {"model__C":[1e-4,1e-3,1e-2,0.1,1,10]},
    "RF": {'model__max_depth':[2,3], 'model__min_samples_leaf':[2,3,4], 'model__min_samples_split':[2,3,4], 'model__n_estimators':[50,100]},
    "MLP": {"model__hidden_layer_sizes":[(10,)], "model__alpha": [0.001,0.01,0.1,1], 'model__max_iter':[2000]},
    "SVM": {"model__C":[1e-3,0.01,0.1,1], 'model__kernel':['rbf','linear'], 'model__gamma':[0.01,0.1,1,10,100]},
    "AB": {'model__learning_rate':[0.001,0.01,0.1], 'model__estimator':[DecisionTreeClassifier(max_depth=i) for i in range(2,4)], 'model__n_estimators':[30,50,100]},
    "ET": {'model__max_depth':[2,3], 'model__min_samples_leaf':[3,4,5], 'model__min_samples_split':[2,3,4], 'model__n_estimators':[50,100]},
    "LGBM": {'model__learning_rate':[0.001,0.01,0.1,1], 'model__max_depth':[2,3], 'model__num_leaves':[5,10,20,31], 'model__n_estimators':[30,50,100]}
}

In [6]:
def get_selectors(model):
    """Return a fresh set of feature selectors for the given model.

    clone(model) gives each selector its own independent estimator so fitted
    state never leaks between the selector and the pipeline's final model.
    n_jobs=1 on all selectors prevents nested parallelism with
    RandomizedSearchCV(n_jobs=-1) which owns all cores at the outer level.
    tol=1e-3 lets sequential selectors stop early when marginal gain is tiny.
    """
    selectors = {}

    # RFECV — tree-based / gradient-boosted models only
    if isinstance(model, (RandomForestClassifier, ExtraTreesClassifier,
                          XGBClassifier, LGBMClassifier)):
        selectors["rfecv"] = RFECV(
            estimator=clone(model), step=1, cv=3,
            scoring="f1_weighted", min_features_to_select=3, n_jobs=1,
        )

    # Sequential selectors — tol stops when marginal F1 gain < 0.1 %
    selectors["ffs"] = SequentialFeatureSelector(
        clone(model), direction="forward", scoring="f1_weighted",
        cv=3, tol=1e-3, n_jobs=1,
    )
    selectors["bfs"] = SequentialFeatureSelector(
        clone(model), direction="backward", scoring="f1_weighted",
        cv=3, tol=1e-3, n_jobs=1,
    )

    selectors["mrmr"] = MRMRSelector(K=10)
    selectors["none"] = "passthrough"

    return selectors

In [7]:
def evaluate(model, X, y):
    y_pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X)
        if y_proba.shape[1] == 2:
            auc = roc_auc_score(y, y_proba[:,1])
        else:
            auc = roc_auc_score(y, y_proba, multi_class="ovr")
    else:
        auc = np.nan

    return {
        "f1_weighted": f1_score(y, y_pred, average="weighted"),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "auc": auc,
        "sensitivity": recall_score(y, y_pred),
        "specificity": recall_score(y, y_pred, pos_label=0)
    }

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

os.makedirs("Results", exist_ok=True)
results = []

for model_name, model in models.items():
    param_grid = param_grids[model_name]
    print(f"\nModel: {model_name}")

    for fs_name, fs in get_selectors(model).items():
        model_path = f"Results/{model_name}_{fs_name}.joblib"

        if os.path.exists(model_path):
            print(f"  FS: {fs_name}  [skipped — already exists]")
            continue

        print(f"  FS: {fs_name}")

        pipe = Pipeline([("fs", fs), ("model", model)])

        search = RandomizedSearchCV(
            pipe,
            param_distributions=param_grid,
            n_iter=20,
            cv=cv,
            scoring="f1_weighted",
            n_jobs=-1,
            random_state=42,
        )

        try:
            search.fit(X_train, y_train, model__sample_weight=sample_weights)
        except Exception:
            search.fit(X_train, y_train)
        
        # Best estimator
        best_model = search.best_estimator_

        n_feats = best_model.named_steps["model"].n_features_in_
        if fs_name != "none":
            print(f"    Number of selected features: {n_feats}")
        
        hyperp = str(search.best_params_)
        print(f"    Hyperparameters of the best estimator:\n    {hyperp}")
        
        cv_f1_mean = search.best_score_
        cv_f1_std = search.cv_results_['std_test_score'][search.best_index_]

        # Evaluation
        train_scores = evaluate(best_model, X_train, y_train)
        test_scores  = evaluate(best_model, X_test,  y_test)

        print("    f1 score:")
        print(f"     - cv: {cv_f1_mean} ({cv_f1_std})")
        print(f"     - train: {train_scores['f1_weighted']}")
        print(f"     - test: {test_scores['f1_weighted']}")

        results.append({
            "model": model_name,
            "fs": fs_name,
            "n features": n_feats,
            "hyperparameters": hyperp,
            "cv_f1_mean": cv_f1_mean,
            "cv_f1_std": cv_f1_std,
            **{f"train {k}": v for k, v in train_scores.items()},
            **{f"test {k}": v for k, v in test_scores.items()},
        })

        # Save search and model
        #joblib.dump(search, f"Results/search_{model_name}_{fs_name}.joblib")
        joblib.dump(best_model, model_path)

        print("\n")
        # Incremental save
        pd.DataFrame(results).to_excel("Results/classif_results.xlsx", index=False)

print("\nDone.")


Model: XGB
  FS: rfecv
    Number of selected features: 4
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.03}
    f1 score:
     - cv: 0.6111359525597932 (0.054879983058644465)
     - train: 0.6877232980488359
     - test: 0.5906200751606255


  FS: ffs
    Number of selected features: 1
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 3, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__eta': 0.03}
    f1 score:
     - cv: 0.6182286664845213 (0.054816521845355674)
     - train: 0.6088806116802642
     - test: 0.62434872232292


  FS: bfs
    Number of selected features: 21
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__eta': 0.03}
    f1 score:
     - cv: 0.6112264867860786 (0.042980358530841076)
     - train: 0.7532728808176109
     - test: 0.5998547087570995


  FS: mrmr


100%|██████████| 10/10 [00:04<00:00,  2.33it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 3, 'model__n_estimators': 30, 'model__max_depth': 2, 'model__eta': 0.3}
    f1 score:
     - cv: 0.610750564443898 (0.037973262727574085)
     - train: 0.7988098986817488
     - test: 0.618045155434536


  FS: none
    Hyperparameters of the best estimator:
    {'model__reg_lambda': 8, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__eta': 0.3}
    f1 score:
     - cv: 0.6139665204996255 (0.052593076002924566)
     - train: 0.9069788234284402
     - test: 0.5650594660403255



Model: LR
  FS: ffs
    Number of selected features: 4
    Hyperparameters of the best estimator:
    {'model__C': 1}
    f1 score:
     - cv: 0.6232210217118695 (0.020252784802233758)
     - train: 0.6569815406256635
     - test: 0.6512821003971446


  FS: bfs
    Number of selected features: 18
    Hyperparameters of the best estimator:
    {'model__C': 0.001}
    f1 score:
     - cv: 0.64837565893

100%|██████████| 10/10 [00:04<00:00,  2.25it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__C': 0.0001}
    f1 score:
     - cv: 0.6406067236339589 (0.04034986344444683)
     - train: 0.6546843294724245
     - test: 0.658577440763288


  FS: none
    Hyperparameters of the best estimator:
    {'model__C': 0.001}
    f1 score:
     - cv: 0.6420677695943414 (0.0306690293129338)
     - train: 0.6679347115730715
     - test: 0.6572862095318263



Model: RF
  FS: rfecv
    Number of selected features: 19
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 3, 'model__min_samples_leaf': 3, 'model__max_depth': 2}
    f1 score:
     - cv: 0.6383451313540318 (0.05004005026037795)
     - train: 0.703863936701962
     - test: 0.6520475728322604


  FS: ffs
    Number of selected features: 5
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__min_samples_split': 2, 'model__min_samples_leaf': 4, 'model__max_depth': 3}

...

    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__n_estimators': 30, 'model__learning_rate': 0.1, 'model__estimator': DecisionTreeClassifier(max_depth=2)}
    f1 score:
     - cv: 0.6182075265244985 (0.05287065807518869)
     - train: 0.68751226028139
     - test: 0.5918914018139994


  FS: none
    Hyperparameters of the best estimator:
    {'model__n_estimators': 30, 'model__learning_rate': 0.1, 'model__estimator': DecisionTreeClassifier(max_depth=3)}
    f1 score:
     - cv: 0.6015535206940332 (0.009598246189729093)
     - train: 0.779070990558953
     - test: 0.5924163394479283



Model: ET
  FS: rfecv
    Number of selected features: 3
    Hyperparameters of the best estimator:
    {'model__n_estimators': 50, 'model__min_samples_split': 4, 'model__min_samples_leaf': 3, 'model__max_depth': 2}
    f1 score:
     - cv: 0.6271773384090839 (0.03871152039313375)
     - train: 0.6377469580797143
     - test: 0.6503900101437616


  FS: ffs
    Num

100%|██████████| 10/10 [00:02<00:00,  4.04it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 3, 'model__min_samples_leaf': 5, 'model__max_depth': 2}
    f1 score:
     - cv: 0.6162935614545951 (0.029726782128144683)
     - train: 0.6459783748203246
     - test: 0.6460176991150443


  FS: none
    Hyperparameters of the best estimator:
    {'model__n_estimators': 100, 'model__min_samples_split': 4, 'model__min_samples_leaf': 3, 'model__max_depth': 2}
    f1 score:
     - cv: 0.6477912599528584 (0.02688032617139542)
     - train: 0.6803682276930872
     - test: 0.6221360771761161



Model: LGBM
  FS: rfecv
    Number of selected features: 8
    Hyperparameters of the best estimator:
    {'model__num_leaves': 10, 'model__n_estimators': 50, 'model__max_depth': 2, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.5928103720675721 (0.04305518258014059)
     - train: 0.7682260395559375
     - test: 0.6094079856064737


  FS: ffs
    Number 

100%|██████████| 10/10 [00:02<00:00,  3.85it/s]


    Number of selected features: 10
    Hyperparameters of the best estimator:
    {'model__num_leaves': 10, 'model__n_estimators': 50, 'model__max_depth': 2, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.6009908087609669 (0.05227987973990188)
     - train: 0.7763078554761521
     - test: 0.6141122511980713


  FS: none
    Hyperparameters of the best estimator:
    {'model__num_leaves': 5, 'model__n_estimators': 50, 'model__max_depth': 3, 'model__learning_rate': 0.1}
    f1 score:
     - cv: 0.6186277316624001 (0.05521806337073851)
     - train: 0.8494223406959129
     - test: 0.5629875126797019



Done.
